In [2]:
import pandas as pd

matches = pd.read_csv("data/ipl_matches.csv")
balls = pd.read_csv("data/ipl_balls.csv")

print(matches.head())
print(balls.head())

   match_id                        team1                  team2  \
0    598067                Pune Warriors       Delhi Daredevils   
1    829725              Kings XI Punjab       Delhi Daredevils   
2    501255  Royal Challengers Bangalore  Kolkata Knight Riders   
3   1082637              Kings XI Punjab          Gujarat Lions   
4    419157  Royal Challengers Bangalore         Mumbai Indians   

                        winner                  toss_winner toss_decision  \
0                Pune Warriors                Pune Warriors           bat   
1             Delhi Daredevils              Kings XI Punjab           bat   
2  Royal Challengers Bangalore  Royal Challengers Bangalore         field   
3                Gujarat Lions                Gujarat Lions         field   
4               Mumbai Indians  Royal Challengers Bangalore         field   

                                               venue        date  
0                         Subrata Roy Sahara Stadium  2013-05-19  


In [4]:
import pandas as pd
import numpy as np

matches_df = pd.read_csv("data/ipl_matches.csv")
balls_df = pd.read_csv("data/ipl_balls.csv")

# Ensure types
matches_df["match_id"] = matches_df["match_id"].astype(str)
balls_df["match_id"] = balls_df["match_id"].astype(str)

# Bring team1/team2 onto balls
balls = balls_df.merge(
    matches_df[["match_id", "team1", "team2"]],
    on="match_id",
    how="left"
)

# Infer bowling_team
balls["bowling_team"] = np.where(
    balls["batting_team"] == balls["team1"],
    balls["team2"],
    balls["team1"]
)

# Sort key inside innings
balls["ball_innings_index"] = balls["over"] * 6 + (balls["ball"] - 1)

balls = balls.sort_values(["match_id", "innings", "ball_innings_index"]).reset_index(drop=True)

# Innings cumulative runs and wickets
balls["runs_so_far"] = balls.groupby(["match_id", "innings"])["total_runs"].cumsum()
balls["wickets_lost"] = balls.groupby(["match_id", "innings"])["wicket"].cumsum()

# Simple momentum: runs in last 12 balls (based on your current ball indexing)
# (Later you can switch to legal-ball based rolling if you parse wide/noball flags.)
balls["runs_last_12"] = (
    balls.groupby(["match_id", "innings"])["total_runs"]
         .rolling(12, min_periods=1)
         .sum()
         .reset_index(level=[0,1], drop=True)
)

# Save processed
balls.to_csv("data/balls_with_state_basic.csv", index=False)

print(balls[[
    "match_id","innings","over","ball","batting_team","bowling_team",
    "total_runs","wicket","runs_so_far","wickets_lost","runs_last_12"
]].head(20))

   match_id  innings  over  ball         batting_team  \
0   1082591        1     0     1  Sunrisers Hyderabad   
1   1082591        1     0     2  Sunrisers Hyderabad   
2   1082591        1     0     3  Sunrisers Hyderabad   
3   1082591        1     0     4  Sunrisers Hyderabad   
4   1082591        1     0     5  Sunrisers Hyderabad   
5   1082591        1     0     6  Sunrisers Hyderabad   
6   1082591        1     0     7  Sunrisers Hyderabad   
7   1082591        1     1     1  Sunrisers Hyderabad   
8   1082591        1     1     2  Sunrisers Hyderabad   
9   1082591        1     1     3  Sunrisers Hyderabad   
10  1082591        1     1     4  Sunrisers Hyderabad   
11  1082591        1     1     5  Sunrisers Hyderabad   
12  1082591        1     1     6  Sunrisers Hyderabad   
13  1082591        1     1     7  Sunrisers Hyderabad   
14  1082591        1     2     1  Sunrisers Hyderabad   
15  1082591        1     2     2  Sunrisers Hyderabad   
16  1082591        1     2     

In [5]:
assert balls["team1"].notna().all()
assert balls["team2"].notna().all()

In [6]:
innings1_final = (
    balls[balls["innings"] == 1]
    .groupby("match_id")["runs_so_far"]
    .max()
)
print(innings1_final.describe())

count    1169.000000
mean      167.021386
std        32.948414
min        56.000000
25%       147.000000
50%       167.000000
75%       188.000000
max       287.000000
Name: runs_so_far, dtype: float64


In [7]:
assert (balls["wickets_lost"] <= 10).all()

In [9]:
import pandas as pd

balls = pd.read_csv("data/balls_with_state_basic.csv")

# Get innings1 totals
innings1_totals = (
    balls[balls["innings"] == 1]
    .groupby("match_id")["runs_so_far"]
    .max()
    .reset_index()
)

innings1_totals["target"] = innings1_totals["runs_so_far"] + 1
innings1_totals = innings1_totals[["match_id", "target"]]

# Merge target into balls table
balls = balls.merge(innings1_totals, on="match_id", how="left")

# Balls used in innings
balls["balls_used"] = balls["over"] * 6 + balls["ball"]

# Balls remaining (T20 = 120 balls)
balls["balls_remaining"] = 120 - balls["balls_used"]

# Wickets remaining
balls["wickets_remaining"] = 10 - balls["wickets_lost"]

# Runs needed (only meaningful for innings 2)
balls["runs_needed"] = balls["target"] - balls["runs_so_far"]

# Required run rate
balls["required_rr"] = balls["runs_needed"] / (balls["balls_remaining"] / 6)

# Clean impossible values
balls.loc[balls["innings"] == 1, ["runs_needed", "required_rr"]] = None

print(balls.head(20))

    match_id  innings  over  ball         batting_team       batsman  \
0    1082591        1     0     1  Sunrisers Hyderabad     DA Warner   
1    1082591        1     0     2  Sunrisers Hyderabad     DA Warner   
2    1082591        1     0     3  Sunrisers Hyderabad     DA Warner   
3    1082591        1     0     4  Sunrisers Hyderabad     DA Warner   
4    1082591        1     0     5  Sunrisers Hyderabad     DA Warner   
5    1082591        1     0     6  Sunrisers Hyderabad      S Dhawan   
6    1082591        1     0     7  Sunrisers Hyderabad      S Dhawan   
7    1082591        1     1     1  Sunrisers Hyderabad      S Dhawan   
8    1082591        1     1     2  Sunrisers Hyderabad     DA Warner   
9    1082591        1     1     3  Sunrisers Hyderabad     DA Warner   
10   1082591        1     1     4  Sunrisers Hyderabad     DA Warner   
11   1082591        1     1     5  Sunrisers Hyderabad     DA Warner   
12   1082591        1     1     6  Sunrisers Hyderabad  MC Henri

In [10]:
balls[balls["innings"] == 2][[
    "runs_so_far",
    "target",
    "runs_needed",
    "balls_remaining",
    "required_rr"
]].head(10)

,runs_so_far,target,runs_needed,balls_remaining,required_rr
125,1,208,207.0,119,10.436975
126,1,208,207.0,118,10.525424
127,1,208,207.0,117,10.615385
128,3,208,205.0,116,10.603448
129,7,208,201.0,115,10.486957
130,11,208,197.0,114,10.368421
131,11,208,197.0,113,10.460177
132,11,208,197.0,112,10.553571
133,12,208,196.0,111,10.594595
134,12,208,196.0,110,10.690909
